# Problem 610 - Roman Numerals II
A random generator produces a sequence of symbols drawn from the set {I, V, X, L, C, D, M, #}. Each item in the sequence is determined by selecting one of these symbols at random, independently of the other items in the sequence. At each step, the seven letters are equally likely to be selected, with probability 14% each, but the # symbol only has a 2% chance of selection.

We write down the sequence of letters from left to right as they are generated, and we stop at the first occurrence of the # symbol (without writing it). However, we stipulate that what we have written down must always (when non-empty) be a valid Roman numeral representation in minimal form. If appending the next letter would contravene this then we simply skip it and try again with the next symbol generated.

Please take careful note of About... Roman Numerals for the definitive rules for this problem on what constitutes a "valid Roman numeral representation" and "minimal form". For example, the (only) sequence that represents 49 is XLIX. The subtractive combination IL is invalid because of rule (ii), while XXXXIX is valid but not minimal. The rules do not place any restriction on the number of occurrences of M, so all positive integers have a valid representation. These are the same rules as were used in Problem 89, and members are invited to solve that problem first.

Find the expected value of the number represented by what we have written down when we stop. (If nothing is written down then count that as zero.) Give your answer rounded to 8 places after the decimal point.


## Solution.
Illegal letters are skipped and re-drawn, so whenever exactly $k$ letters may legally be appended, each of them is written with conditional probability $p(k)=\frac{14}{14k+2}$.
All seven letters stay legal while the $M$'s are written, so their number is geometric with mean $p(7)/(1-p(7))$, and the hundreds, tens and units groups follow, each entered conditionally on the letters already passed over no longer being legal.
Their distributions are then given by one and the same formula with the index triples $(6,7,5)$, $(4,5,3)$ and $(2,3,1)$, counting the legal letters at the start of the group, after a single $C$ (resp. $X$, $I$) and after $D$ or $CC$ (resp. $L$ or $XX$, $V$ or $II$).


In [1]:
from functools import cache

In [2]:
def roman_to_arabic(m):
    ans = 0

    patterns = {'CD':400, 'CM': 900, 'XL': 40, 'XC': 90, 'IV': 4, 'IX':9}
    values = {'I':1, 'V':5, 'X':10, 'L':50, 'C':100, 'D':500, 'M':1000}

    i = 0
    while i < len(m):
        j = i

        while j < len(m) and m[i] == m[j]:
            j += 1

        if m[i:(j+1)] in patterns:
            ans += patterns[m[i:(j+1)]]
            i = j+1
        else:
            ans += values[m[i]] * (j-i)
            i = j

    return ans

In [5]:
def p(k):
    return 14 / (14*k + 2)


def q(k):
    return 2 / (14*k + 2)


def expectation(group):
    return sum(roman_to_arabic(s) * prob for s, prob in group.items())


In [6]:
# the three groups share one formula, with the index triples (6,7,5), (4,5,3), (2,3,1)

hundreds = {
    "":     1 - 2*p(6),                 # 0
    "C":    p(6) * (1 - 3*p(7)),        # 1
    "CC":   p(6) * p(7) * (1 - p(5)),   # 2
    "CCC":  p(6) * p(7) * p(5),         # 3
    "CD":   p(6) * p(7),                # 4
    "D":    p(6) * (1 - p(5)),          # 5
    "DC":   p(6) * p(5) * (1 - p(5)),   # 6
    "DCC":  p(6) * p(5)**2 * (1 - p(5)),# 7
    "DCCC": p(6) * p(5)**3,             # 8
    "CM":   p(6) * p(7),                # 9
}

tens = {
    "":     1 - 2*p(4),
    "X":    p(4) * (1 - 3*p(5)),
    "XX":   p(4) * p(5) * (1 - p(3)),
    "XXX":  p(4) * p(5) * p(3),
    "XL":   p(4) * p(5),
    "L":    p(4) * (1 - p(3)),
    "LX":   p(4) * p(3) * (1 - p(3)),
    "LXX":  p(4) * p(3)**2 * (1 - p(3)),
    "LXXX": p(4) * p(3)**3,
    "XC":   p(4) * p(5),
}

units = {
    "":     1 - 2*p(2),
    "I":    p(2) * (1 - 3*p(3)),
    "II":   p(2) * p(3) * (1 - p(1)),
    "III":  p(2) * p(3) * p(1),
    "IV":   p(2) * p(3),
    "V":    p(2) * (1 - p(1)),
    "VI":   p(2) * p(1) * (1 - p(1)),
    "VII":  p(2) * p(1)**2 * (1 - p(1)),
    "VIII": p(2) * p(1)**3,
    "IX":   p(2) * p(3),
}


In [9]:
e = 1000 * p(7)/(1 - p(7))

e += expectation(hundreds) + expectation(tens) + expectation(units)

print(f"Expectation is {round(e, 8)}")


Expectation is 319.30207833
